# simple-2048 RL on Kaggle

Faithful PyTorch recreation of [kywch's *Less is More: 2048 agents*](https://kywch.github.io/blog/2026/01/less-is-more-2048-agents/) (`simple-2048` branch of PufferLib, MIT License).

**Setup:** set `REPO_URL` in the cell below to your GitHub repo and turn on a **GPU** accelerator (T4/P100). Notebook settings must have **Internet: on** (needed for both the clone and `pip install heavyball`). Private repo? Use `https://<TOKEN>@github.com/you/repo.git` via Kaggle Secrets rather than pasting a token in plain text.

No-internet sessions only: attach the project as a Kaggle Dataset instead and leave `REPO_URL` empty — the setup cell falls back to searching `/kaggle/input` (you would also need heavyball as an offline wheel).

Each session: run all cells. The training cell automatically resumes from the newest checkpoint in `/kaggle/working/experiments` (checkpoints live outside the clone, so re-cloning never touches them).

In [ ]:
# 1) Dependencies. torch/numpy are preinstalled on Kaggle;
#    heavyball 2.1.4 provides the original ForeachMuon optimizer.
%pip install -q heavyball==2.1.4
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 2) Get the project: clone/pull your GitHub repo (or fall back to a
#    Kaggle Dataset under /kaggle/input if REPO_URL is left empty).
REPO_URL = 'https://github.com/m3likaj/2048PPO'   # <-- e.g. 'https://github.com/<you>/<repo>.git'

import os, glob, subprocess
ROOT = '/kaggle/working/repo'
if REPO_URL:
    if os.path.exists(os.path.join(ROOT, '.git')):
        subprocess.run(['git', '-C', ROOT, 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, ROOT], check=True)
    base = ROOT
else:
    base = '/kaggle/input'   # dataset fallback (no-internet sessions)

hits = glob.glob(f'{base}/**/train.py', recursive=True)
hits = [h for h in hits if os.path.exists(os.path.join(os.path.dirname(h), 'g2048_env.py'))]
assert hits, 'Project not found: set REPO_URL or attach the project as a Dataset.'
PROJ = os.path.dirname(hits[0])
os.chdir(PROJ)
print('Working dir:', os.getcwd())

In [ ]:
# 3) Exactness tests (~2 min). All 14 must pass before training.
!python test_exactness.py

In [ ]:
# 4) Train (auto-resumes). All hyperparameters are the originals; only the
#    scale knobs below are yours to set. batch auto-resolves to NUM_ENVS*64.
#    Full original config: NUM_ENVS=16384, TOTAL_TIMESTEPS=6_767_676_767
#    (multi-day on a T4 -- train it in resumable chunks like this).
import glob, os

NUM_ENVS = 4096            # T4-friendly; use 16384 to match the original exactly
TOTAL_TIMESTEPS = 1e9      # raise toward 6.77e9 across sessions
DATA_DIR = '/kaggle/working/experiments'

ckpts = sorted(glob.glob(f'{DATA_DIR}/*/latest.pt'), key=os.path.getmtime)
resume = f'--resume {ckpts[-1]}' if ckpts else ''
print('Resuming from:', ckpts[-1] if ckpts else '(fresh run)')

!python train.py --num-envs {NUM_ENVS} --total-timesteps {TOTAL_TIMESTEPS} \
    --data-dir {DATA_DIR} --tag kaggle {resume}

In [ ]:
# 5) Evaluate the newest checkpoint (original protocol: scaffolding = 0).
#    Reference from the original run: 84.88% reach 32768, 33.96% reach 65536.
import glob, os
ckpts = sorted(glob.glob('/kaggle/working/experiments/*/latest.pt'), key=os.path.getmtime)
assert ckpts, 'Train first.'
!python eval.py --checkpoint {ckpts[-1]} --num-envs 4096 --min-episodes 5000

In [ ]:
# 6) Progress report + article-ready plots for the newest run.
import glob, os
from IPython.display import Image, display
runs = sorted(glob.glob('/kaggle/working/experiments/*/log.csv'), key=os.path.getmtime)
assert runs, 'Train first.'
run_dir = os.path.dirname(runs[-1])
print(open(os.path.join(run_dir, 'report.txt')).read()[-2500:])   # latest report blocks
!python plot_training.py --csv {runs[-1]}
for name in ['milestone_8192.png', 'milestone_16384.png', 'milestone_32768.png', 'losses.png', 'explained_variance.png', 'avg_max_tile.png']:
    p = os.path.join(run_dir, 'plots', name)
    if os.path.exists(p):
        display(Image(p))

**Notes**
- Checkpoints: `/kaggle/working/experiments/<run>/latest.pt` (full state) and `model_XXXXXX.pt` every 200 epochs; `log.csv` has per-epoch metrics + `x/...` earned/scaffold columns; `report.txt` gets ~20 progress blocks per run (timestamps, episode counts, EV, earned 8k/16k/32k % **with raw counts**); `plots/` holds the graphs. Save the notebook version so outputs persist, then next session the train cell resumes automatically (report numbering and cumulative counts continue).
- `--optimizer adam` is available as a fallback, but muon (default) is the original.
- Transformer experiments later: swap `G2048Policy.encoder` / `encode_observations` in `g2048_policy.py` (see README).